In [ ]:
!pip install -q requests pandas openpyxl tabulate

import json
import re
import time
import requests
import pandas as pd
from IPython.display import display

# ==============================================================================
# 1. Connection & Resilient Utility Functions
# ==============================================================================

# Update with your active ngrok URL forwarding to LM Studio (port 1234)
LM_STUDIO_URL = "https://spirits-copartner-unsorted.ngrok-free.dev"

def get_clean_endpoint(url: str) -> str:
    url = url.strip().rstrip("/")
    if not url.endswith("/v1/chat/completions"):
        url = f"{url}/v1/chat/completions"
    return url

def query_local_llm(system_prompt: str, user_prompt: str, retries: int = 3, delay: int = 4) -> str:
    """Sends prompt to LM Studio with headers and retry logic for network drops."""
    endpoint = get_clean_endpoint(LM_STUDIO_URL)
    payload = {
        "model": "local-model",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.2
    }
    headers = {
        "Content-Type": "application/json",
        "ngrok-skip-browser-warning": "true",
        "User-Agent": "ColabClient"
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(endpoint, headers=headers, json=payload, timeout=180)
            if response.status_code == 200:
                res_json = response.json()
                if "choices" in res_json and len(res_json["choices"]) > 0:
                    return res_json["choices"][0]["message"]["content"]
                raise KeyError(f"'choices' missing in response:\n{json.dumps(res_json, indent=2)}")

            print(f"  [Attempt {attempt}/{retries}] Server dropped (HTTP {response.status_code}). Retrying in {delay}s...")
            time.sleep(delay)
        except (requests.exceptions.RequestException, KeyError) as e:
            if attempt == retries:
                raise RuntimeError(f"Failed after {retries} attempts: {e}")
            print(f"  [Attempt {attempt}/{retries}] Connection issue: {e}. Retrying in {delay}s...")
            time.sleep(delay)

    raise RuntimeError("Unexpected failure in query_local_llm.")

def clean_json_output(text: str):
    """
    Robustly extracts JSON from raw LLM responses.
    Handles markdown fences, embedded lists, and concatenated objects ({...}{...}).
    """
    text = text.strip()
    text = re.sub(r"```(?:json)?", "", text)
    text = re.sub(r"```", "", text).strip()

    # 1. Standard single JSON parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Extract embedded bracketed array [...]
    match = re.search(r"(\[.*\])", text, re.DOTALL)
    if match:
        clean_text = match.group(1).strip()
        clean_text = re.sub(r",\s*([\]\}])", r"\1", clean_text)
        try:
            return json.loads(clean_text)
        except json.JSONDecodeError:
            pass

    # 3. Fallback: Parse stream-concatenated JSON objects ({...}{...}{...})
    objects = []
    decoder = json.JSONDecoder()
    pos = 0
    while pos < len(text):
        m = re.search(r"\{", text[pos:])
        if not m:
            break
        start_idx = pos + m.start()
        try:
            obj, end_offset = decoder.raw_decode(text[start_idx:])
            objects.append(obj)
            pos = start_idx + end_offset
        except json.JSONDecodeError:
            pos = start_idx + 1

    if objects:
        return objects

    print("\n--- RAW LLM RESPONSE CAUSING ERROR ---")
    print(text)
    print("--------------------------------------\n")
    raise ValueError("Could not parse any valid JSON objects from model output.")

def extract_assembly_name(component_spec: str) -> str:
    match = re.search(r"Component\s*:\s*([^\n\r.]+)", component_spec, re.IGNORECASE)
    return match.group(1).strip() if match else "Mechanical Assembly"

# ==============================================================================
# 2. Multi-Agent Pipeline (Part-Level Granularity with Safe Type Casting)
# ==============================================================================

def profiler_agent(component_spec: str) -> list:
    """Stage 1: Decomposes assembly into parts and their respective failure modes."""
    system_prompt = (
        "You are an expert Mechanical Reliability Engineer acting as an FMEA Profiler. "
        "Adhere strictly to AIAG-VDA and SAE J1739 standards. "
        "Analyze the assembly and identify 3 to 5 critical parts/sub-assemblies. "
        "For each part, identify its primary failure mode, potential cause, and effect. "
        "CRITICAL: Do NOT use numeric indexes or component numbers. Provide the explicit component name (e.g. 'Threaded Spindle', 'Ductile Iron Nut'). "
        "Return ONLY a raw JSON ARRAY enclosed in square brackets [...] of objects with keys: "
        "'part_name', 'failure_mode', 'cause', 'effect'."
    )
    raw_response = query_local_llm(system_prompt, f"Specification: {component_spec}")
    data = clean_json_output(raw_response)
    if isinstance(data, dict):
        data = [data]
    return [item for item in data if isinstance(item, dict)]

def evaluator_agent(failure_modes: list) -> list:
    """Stage 2: Evaluator Agent scores S, O, D (1-10) without computing RPN."""
    system_prompt = (
        "You are an FMEA Risk Evaluator adhering strictly to AIAG-VDA guidelines. "
        "Score each part's failure mode on a scale of 1 to 10 on:\n"
        "- severity: 1 (negligible) to 10 (hazardous without warning)\n"
        "- occurrence: 1 (extremely unlikely) to 10 (inevitable)\n"
        "- detection: 1 (almost certain detection) to 10 (undetectable)\n"
        "Retain the exact string 'part_name' from the input (do not replace with numeric IDs). "
        "Return ONLY a raw JSON list of updated objects with keys: "
        "'part_name', 'failure_mode', 'cause', 'effect', 'severity', 'occurrence', 'detection'. "
        "DO NOT calculate RPN."
    )
    raw_response = query_local_llm(system_prompt, json.dumps(failure_modes))
    data = clean_json_output(raw_response)
    if isinstance(data, dict):
        data = [data]
    return [item for item in data if isinstance(item, dict)]

def python_rpn_guardrail(scored_modes: list) -> list:
    """Stage 3: Deterministic calculation of RPN = Severity * Occurrence * Detection."""
    for row in scored_modes:
        try:
            s = int(row.get("severity", 1))
        except (ValueError, TypeError):
            s = 1
        try:
            o = int(row.get("occurrence", 1))
        except (ValueError, TypeError):
            o = 1
        try:
            d = int(row.get("detection", 1))
        except (ValueError, TypeError):
            d = 1
        row["severity"] = s
        row["occurrence"] = o
        row["detection"] = d
        row["rpn"] = s * o * d
    return scored_modes

def mitigation_agent(analyzed_modes: list, rpn_threshold: int = 100) -> list:
    """
    Stage 4: Mitigation Agent provides redesign solutions.
    Applies AIAG-VDA Action Priority: triggers if RPN >= threshold OR Severity >= 9.
    """
    high_risk_items = [
        item for item in analyzed_modes
        if int(item.get("rpn", 0)) >= rpn_threshold or int(item.get("severity", 1)) >= 9
    ]

    for item in analyzed_modes:
        item["mitigation"] = "Risk acceptable (Standard preventive maintenance/monitoring)"

    if not high_risk_items:
        return analyzed_modes

    system_prompt = (
        "You are a Senior Reliability and Mitigation Engineer. "
        "Provide specific engineering actions (material change, geometric redesign, lubrication; NOT schedule checks) "
        "for the high-risk parts per AIAG-VDA. "
        "Return ONLY a raw JSON list of objects where each object has keys: "
        "'part_name', 'failure_mode', 'mitigation'."
    )
    raw_response = query_local_llm(system_prompt, json.dumps(high_risk_items))
    mitigations = clean_json_output(raw_response)

    if isinstance(mitigations, dict):
        for key in ["mitigations", "actions", "results", "items"]:
            if key in mitigations and isinstance(mitigations[key], list):
                mitigations = mitigations[key]
                break
        else:
            mitigations = [mitigations]

    mitigation_dict = {}
    if isinstance(mitigations, list):
        for m in mitigations:
            if isinstance(m, dict):
                p_name = str(m.get("part_name", "")).strip().lower()
                f_mode = str(m.get("failure_mode", "")).strip().lower()
                action = str(m.get("mitigation") or m.get("action") or m.get("recommended_action") or "").strip()
                if f_mode and action:
                    mitigation_dict[(p_name, f_mode)] = action
                    mitigation_dict[f_mode] = action

    for item in analyzed_modes:
        item_part = str(item.get("part_name", "")).strip().lower()
        item_mode = str(item.get("failure_mode", "")).strip().lower()

        if (item_part, item_mode) in mitigation_dict:
            item["mitigation"] = mitigation_dict[(item_part, item_mode)]
        elif item_mode in mitigation_dict:
            item["mitigation"] = mitigation_dict[item_mode]

    return analyzed_modes

# ==============================================================================
# 3. Execution & Export
# ==============================================================================

def run_fmea_pipeline(component_spec: str, rpn_threshold: int = 100) -> pd.DataFrame:
    print("=" * 80)
    print("RUNNING PART-BY-PART MULTI-AGENT FMEA")
    print("=" * 80)

    assembly_name = extract_assembly_name(component_spec)

    print("\n[Stage 1/4] Profiling assembly into individual parts and failure modes...")
    raw_failures = profiler_agent(component_spec)

    print("[Stage 2/4] Rating Severity (S), Occurrence (O), and Detection (D)...")
    scored_failures = evaluator_agent(raw_failures)

    print("[Stage 3/4] Deterministic Python Guardrail (Computing RPN)...")
    guarded_failures = python_rpn_guardrail(scored_failures)

    print(f"[Stage 4/4] Mitigation Agent (Engineering actions for RPN >= {rpn_threshold} or S >= 9)...")
    final_results = mitigation_agent(guarded_failures, rpn_threshold=rpn_threshold)

    for row in final_results:
        row["assembly"] = assembly_name

    df = pd.DataFrame(final_results)

    column_mapping = {
        "assembly": "Assembly",
        "part_name": "Part / Component",
        "failure_mode": "Potential Failure Mode",
        "cause": "Potential Root Cause",
        "effect": "Local / System Effect",
        "severity": "S",
        "occurrence": "O",
        "detection": "D",
        "rpn": "RPN",
        "mitigation": "Recommended Engineering Actions"
    }

    present_cols = [k for k in column_mapping.keys() if k in df.columns]
    df = df[present_cols].rename(columns=column_mapping)

    if "RPN" in df.columns:
        df = df.sort_values(by="RPN", ascending=False).reset_index(drop=True)

    return df

# ==============================================================================
# 4. Batch Demonstration Run
# ==============================================================================

test_cases = {
    "Toggle Screw Jack": (
        "Component: Toggle Screw Jack Assembly\n"
        "Application: Automotive garage vehicle lifting\n"
        "Operating Load: 15 kN static, cyclic lifting loading\n"
        "Material: Mild steel threaded spindle with ductile iron nut, alloy steel toggle pins, and cast steel base."
    ),
    "Hydraulic Bottle Jack": (
        "Component: 5-Ton Hydraulic Bottle Jack Assembly\n"
        "Application: Heavy commercial vehicle roadside maintenance\n"
        "Operating Load: 50 kN hydraulic pressure (approx. 350 bar peak)\n"
        "Material: Chrome-plated forged steel piston rod, cast iron base reservoir, nitrile elastomeric seals, and hardened steel release valve."
    ),
    "Worm Gear Reducer": (
        "Component: Industrial Worm Gear Speed Reducer\n"
        "Application: Material handling conveyor drive unit\n"
        "Operating Load: 1.5 kW continuous input, 1440 RPM to 48 RPM (30:1 reduction ratio)\n"
        "Material: Hardened 40Cr steel worm shaft, phosphor bronze worm wheel rim, ductile iron casing, and tapered roller bearings."
    ),
    "Brake Caliper": (
        "Component: Floating Single-Piston Brake Caliper Assembly\n"
        "Application: Passenger vehicle front axle service brake\n"
        "Operating Load: Hydraulic line pressure up to 120 bar, thermal cycling up to 450 deg C\n"
        "Material: Nodular cast iron housing, phenolic composite piston, EPDM square-cut seal ring, and stainless steel slider guide pins."
    )
}

all_reports = []

for name, spec in test_cases.items():
    print(f"\n{'='*70}")
    print(f" PROCESSING: {name.upper()}")
    print(f"{'='*70}")

    df = run_fmea_pipeline(spec, rpn_threshold=100)
    all_reports.append(df)

    print(f"\n--- Completed FMEA for {name} ---")
    display(df)

master_fmea_df = pd.concat(all_reports, ignore_index=True)
master_fmea_df.to_csv("Master_FMEA_Consolidated_Report.csv", index=False)
master_fmea_df.to_excel("Master_FMEA_Consolidated_Report.xlsx", index=False)

print(f"\n{'='*70}")
print("ALL RUNS COMPLETE!")
print(f"Total Failure Modes Evaluated: {len(master_fmea_df)}")
print("Master files saved: 'Master_FMEA_Consolidated_Report.csv' and 'Master_FMEA_Consolidated_Report.xlsx'")
print(f"{'='*70}")


 PROCESSING: TOGGLE SCREW JACK
RUNNING PART-BY-PART MULTI-AGENT FMEA

[Stage 1/4] Profiling assembly into individual parts and failure modes...
[Stage 2/4] Rating Severity (S), Occurrence (O), and Detection (D)...
[Stage 3/4] Deterministic Python Guardrail (Computing RPN)...
[Stage 4/4] Mitigation Agent (Engineering actions for RPN >= 100 or S >= 9)...

--- Completed FMEA for Toggle Screw Jack ---


,Assembly,Part / Component,Potential Failure Mode,Potential Root Cause,Local / System Effect,S,O,D,RPN,Recommended Engineering Actions
0,Toggle Screw Jack Assembly,Cast Steel Base,Base cracking or excessive deformation,"Overloading during lifting operation, inadequa...","Loss of lifting capacity, potential vehicle da...",9,8,6,432,"[{'action': 'Geometric redesign', 'description..."
1,Toggle Screw Jack Assembly,Ductile Iron Nut,Nut stripping or rounding,"Insufficient tightening torque, over-tightenin...","Loss of lifting capacity, potential vehicle da...",8,7,5,280,"[{'action': 'Material change', 'description': ..."
2,Toggle Screw Jack Assembly,Mild Steel Threaded Spindle,Spindle fracture or excessive wear,"Over-torquing during lifting operation, inadeq...","Loss of lifting capacity, potential vehicle da...",8,6,4,192,"[{'action': 'Material change', 'description': ..."
3,Toggle Screw Jack Assembly,Alloy Steel Toggle Pin,Pin breakage or excessive wear,"Over-stress during lifting operation, inadequa...","Loss of lifting capacity, potential vehicle da...",9,5,3,135,"[{'action': 'Material change', 'description': ..."



 PROCESSING: HYDRAULIC BOTTLE JACK
RUNNING PART-BY-PART MULTI-AGENT FMEA

[Stage 1/4] Profiling assembly into individual parts and failure modes...
[Stage 2/4] Rating Severity (S), Occurrence (O), and Detection (D)...
[Stage 3/4] Deterministic Python Guardrail (Computing RPN)...
[Stage 4/4] Mitigation Agent (Engineering actions for RPN >= 100 or S >= 9)...

--- Completed FMEA for Hydraulic Bottle Jack ---


,Assembly,Part / Component,Potential Failure Mode,Potential Root Cause,Local / System Effect,S,O,D,RPN,Recommended Engineering Actions
0,5-Ton Hydraulic Bottle Jack Assembly,Release Valve,Inadequate pressure relief leading to over-pre...,"Clogged or damaged valve seat, incorrect valve...","Uncontrolled hydraulic pressure buildup, poten...",9,7,5,315,"[{'action': 'Material change', 'description': ..."
1,5-Ton Hydraulic Bottle Jack Assembly,Cast Iron Base Reservoir,Structural failure due to excessive pressure o...,Overloading the jack beyond its rated capacity...,"Loss of structural integrity, potential collap...",9,6,4,216,"[{'action': 'Material change', 'description': ..."
2,5-Ton Hydraulic Bottle Jack Assembly,Piston Rod,Bending or cracking due to excessive load or i...,Overloading the jack beyond its rated capacity...,"Loss of structural integrity, potential collap...",8,6,4,192,"[{'action': 'Material change', 'description': ..."
3,5-Ton Hydraulic Bottle Jack Assembly,Nitrile Elastomeric Seals,Leakage due to seal degradation or improper in...,"Exposure to extreme temperatures, contaminatio...","Hydraulic fluid leakage, potential system fail...",8,5,3,120,"[{'action': 'Material change', 'description': ..."
4,5-Ton Hydraulic Bottle Jack Assembly,Chrome-Plated Forged Steel Piston Rod Retainer...,Loosening or detachment due to vibration or ex...,"Incorrect clip installation, loose fasteners, ...","Piston rod displacement, potential system fail...",8,5,3,120,"[{'action': 'Material change', 'description': ..."



 PROCESSING: WORM GEAR REDUCER
RUNNING PART-BY-PART MULTI-AGENT FMEA

[Stage 1/4] Profiling assembly into individual parts and failure modes...
[Stage 2/4] Rating Severity (S), Occurrence (O), and Detection (D)...
[Stage 3/4] Deterministic Python Guardrail (Computing RPN)...
[Stage 4/4] Mitigation Agent (Engineering actions for RPN >= 100 or S >= 9)...

--- Completed FMEA for Worm Gear Reducer ---


,Assembly,Part / Component,Potential Failure Mode,Potential Root Cause,Local / System Effect,S,O,D,RPN,Recommended Engineering Actions
0,Industrial Worm Gear Speed Reducer,Worm Shaft,Surface Fatigue Failure,High cyclic stress due to misaligned or worn-o...,"Gear teeth wear out prematurely, causing speed...",8,6,4,192,"[{'action': 'Material Change', 'description': ..."
1,Industrial Worm Gear Speed Reducer,Worm Wheel Rim,Material Fatigue Failure,High cyclic stress due to misaligned or worn-o...,"Gear teeth wear out prematurely, causing speed...",8,6,4,192,"[{'action': 'Geometric Redesign', 'description..."
2,Industrial Worm Gear Speed Reducer,Tapered Roller Bearing,Roller Spalling Failure,Insufficient lubrication or contamination of b...,"Increased vibration and noise, leading to prem...",5,7,3,105,"[{'action': 'Lubrication Change', 'description..."
3,Industrial Worm Gear Speed Reducer,Ductile Iron Casing,Cracking due to Thermal Stress,Inadequate heat dissipation or thermal expansi...,"Casing failure, exposing internal components t...",9,5,2,90,"[{'action': 'Heat Dissipation Enhancement', 'd..."



 PROCESSING: BRAKE CALIPER
RUNNING PART-BY-PART MULTI-AGENT FMEA

[Stage 1/4] Profiling assembly into individual parts and failure modes...
[Stage 2/4] Rating Severity (S), Occurrence (O), and Detection (D)...
[Stage 3/4] Deterministic Python Guardrail (Computing RPN)...
[Stage 4/4] Mitigation Agent (Engineering actions for RPN >= 100 or S >= 9)...

--- Completed FMEA for Brake Caliper ---


,Assembly,Part / Component,Potential Failure Mode,Potential Root Cause,Local / System Effect,S,O,D,RPN,Recommended Engineering Actions
0,Floating Single-Piston Brake Caliper Assembly,Nodular Cast Iron Housing,Cracking due to thermal stress or fatigue,Inadequate heat treatment or material selectio...,"Brake fluid leakage, reduced braking performance",9,7,6,378,Replace the Nodular Cast Iron material with a ...
1,Floating Single-Piston Brake Caliper Assembly,EPDM Square-Cut Seal Ring,Compression set or seal extrusion due to exces...,Insufficient seal material hardness or inadequ...,"Brake fluid leakage, reduced braking performance",9,8,5,360,"Replace the EPDM material with a more durable,..."
2,Floating Single-Piston Brake Caliper Assembly,Piston,Seal failure due to piston thermal expansion m...,Inadequate thermal expansion compensation betw...,"Brake fluid leakage, reduced braking performance",8,6,4,192,Replace piston with a new design featuring a t...
3,Floating Single-Piston Brake Caliper Assembly,Stainless Steel Slider Guide Pin,Pitting or galling due to corrosion or wear,Inadequate surface finish or lubrication of gu...,"Increased brake caliper vibration, reduced bra...",6,5,3,90,Risk acceptable (Standard preventive maintenan...



ALL RUNS COMPLETE!
Total Failure Modes Evaluated: 17
Master files saved: 'Master_FMEA_Consolidated_Report.csv' and 'Master_FMEA_Consolidated_Report.xlsx'
